# Ejercicio 1: Prediccion de Demanda de Bicicletas Compartidas

## Minería de Datos II - Unidad 3: Modelos de Regresión

### Descripcion del problema

Los sistemas de bicicletas compartidas (bike sharing) registran automaticamente el proceso de alquiler y devolucion de bicicletas en distintas estaciones de una ciudad. Conocer con anticipacion cuantas bicicletas se necesitaran en un dia determinado permite a los operadores planificar el mantenimiento, la redistribucion de unidades entre estaciones y el dimensionamiento de la flota.

**Objetivo:** Predecir la cantidad total de alquileres diarios (variable `cnt`) a partir de informacion temporal (estacion del ano, mes, dia de la semana, dia feriado) y meteorologica (temperatura, humedad, velocidad del viento).

### Dataset
- **Nombre:** BikeSharingDataset
- **Fuente:** UCI Machine Learning Repository
- **Registros:** 731 registros diarios (day.csv)
- **Variable objetivo:** `cnt` (cantidad total de bicicletas alquiladas = usuarios casuales + registrados)

### Algoritmos a utilizar
1. **Regresion Lineal Multiple** - Baseline simple e interpretable
2. **Random Forest Regressor** - Captura relaciones no lineales y efectos de interaccion

---
## 1. Importacion de Librerias

Primero importamos todas las librerias necesarias para el analisis:
- `pandas` y `numpy` para manipulacion de datos
- `matplotlib` y `seaborn` para visualizacion
- `sklearn` para los modelos de regresion y metricas de evaluacion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print('Librerias importadas correctamente.')

Se importan todas las librerias necesarias. `sklearn` nos proporciona los algoritmos de regresion lineal multiple y random forest, ainsi como las funciones para dividir los datos y evaluar los modelos.

---
## 2. Carga y Exploracion del Dataset

Cargamos el dataset y realizamos un analisis exploratorio breve:
- Tipos de datos
- Valores faltantes
- Distribucion de la variable objetivo
- Estadisticas descriptivas

In [ ]:
# Descargar y cargar el dataset desde UCI
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip'
import zipfile
import io
import requests

response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))
datos = pd.read_csv(z.open('day.csv'))

print(f'Forma del dataset: {datos.shape}')
print(f'\nPrimeras 5 filas:')
datos.head()

Descargamos el dataset directamente desde el repositorio UCI. El archivo `day.csv` contiene 731 registros diarios con 16 columnas. Cada fila representa un dia del ano con datos de clima, temporal y la cantidad de bicicletas alquiladas.

In [ ]:
# Informacion general del dataset
print('=== INFORMACION DEL DATASET ===')
print(datos.info())
print('\n=== VALORES FALTANTES ===')
print(datos.isnull().sum())
print('\n=== ESTADISTICAS DESCRIPTIVAS ===')
datos.describe()

Observamos que:
- **No hay valores faltantes** en el dataset
- Todas las columnas son numericas (enteros o flotantes)
- La variable objetivo `cnt` tiene un promedio de ~4504 bicicletas alquiladas por dia
- El rango de `cnt` va desde 22 hasta 8714

In [ ]:
# Distribucion de la variable objetivo (cnt)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(datos['cnt'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribucion de Bicicletas Alquiladas (cnt)')
axes[0].set_xlabel('Cantidad de Bicicletas')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(datos['cnt'].mean(), color='red', linestyle='--', label=f'Media: {datos["cnt"].mean():.0f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(datos['cnt'], vert=True)
axes[1].set_title('Boxplot de Bicicletas Alquiladas')
axes[1].set_ylabel('Cantidad de Bicicletas')

plt.tight_layout()
plt.show()

print(f'\nMedia: {datos["cnt"].mean():.2f}')
print(f'Mediana: {datos["cnt"].median():.2f}')
print(f'Desvio Estandar: {datos["cnt"].std():.2f}')
print(f'Minimo: {datos["cnt"].min()}')
print(f'Maximo: {datos["cnt"].max()}')

Visualizamos la distribucion de la variable objetivo `cnt`:
- El **histograma** muestra una distribucion aproximadamente bimodal, lo que sugiere que hay dias con poca demanda (invierno) y dias con mucha demanda (verano)
- El **boxplot** permite identificar la mediana y posibles valores atipicos
- La media es ~4504 y el desvio estandar ~1937, lo que indica variabilidad en la demanda

---
## 3. Analisis de Correlacion

Analizamos la correlacion entre las variables predictoras y la variable objetivo para entender que factores influyen mas en la demanda de bicicletas.

In [ ]:
# Mapa de correlaciones
plt.figure(figsize=(14, 10))
correlacion = datos.corr()
sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Mapa de Correlaciones entre Variables')
plt.tight_layout()
plt.show()

El mapa de correlaciones muestra:
- `temp` (temperatura) y `atemp` (sensacion termica) tienen **alta correlacion positiva** con `cnt` (~0.63)
- `yr` (ano) tiene correlacion positiva (~0.23), indicando crecimiento de la demanda
- `hum` (humedad) y `windspeed` (viento) tienen correlacion negativa con `cnt`
- `season`, `mnth` tambien muestran correlacion positiva con la demanda
- Observamos **multicolinealidad** entre `temp` y `atemp` (correlacion ~0.99), lo que es importante a la hora de seleccionar variables

---
## 4. Preprocesamiento de Datos

Realizamos las siguientes tareas de preprocesamiento:
1. Seleccion de variables predictoras relevantes
2. Division en conjuntos de entrenamiento y prueba (80/20)
3. No es necesario escalar para Regresion Lineal y Random Forest

In [ ]:
# Seleccion de variables predictoras
# Excluimos: instant (indice), dteday (fecha), cnt (objetivo), casual y registrados (desagregados)
columnas_seleccionadas = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday',
                          'weathersit', 'temp', 'atemp', 'hum', 'windspeed']

X = datos[columnas_seleccionadas]
y = datos['cnt']

print(f'Variables predictoras seleccionadas: {columnas_seleccionadas}')
print(f'Forma de X: {X.shape}')
print(f'Forma de y: {y.shape}')

Seleccionamos 11 variables predictoras basadas en el analisis de correlacion y la relevancia para el problema. Excluimos `instant` (indice), `dteday` (fecha), `cnt` (objetivo) y `casual`/`registrados` (desagregados de la variable objetivo).

In [ ]:
# Division en conjuntos de entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Conjunto de entrenamiento: {X_train.shape[0]} registros')
print(f'Conjunto de prueba: {X_test.shape[0]} registros')
print(f'\nProporcion train/test: {X_train.shape[0]/X_test.shape[0]:.1f}/1')

Dividimos los datos en 80% entrenamiento (585 registros) y 20% prueba (146 registros). Usamos `random_state=42` para garantizar la reproducibilidad de los resultados.

---
## 5. Modelo 1: Regresion Lineal Multiple

La regresion lineal multiple modela la relacion entre la variable dependiente y multiples variables independientes mediante una ecuacion lineal:

$$cnt = \beta_0 + \beta_1 \cdot season + \beta_2 \cdot yr + ... + \beta_n \cdot windspeed$$

Es un **baseline simple e interpretable** que permite ver si las relaciones entre variables son aproximadamente lineales.

In [ ]:
# Crear y entrenar el modelo de Regresion Lineal Multiple
modelo_lr = LinearRegression()
modelo_lr.fit(X_train, y_train)

# Predicciones
y_pred_lr = modelo_lr.predict(X_test)

# Coeficientes del modelo
print('=== COEFICIENTES DEL MODELO DE REGRESION LINEAL ===')
print(f'Intercepto: {modelo_lr.intercept_:.4f}')
print('\nCoeficientes por variable:')
for nombre, coef in zip(columnas_seleccionadas, modelo_lr.coef_):
    print(f'  {nombre:15s}: {coef:10.4f}')

Los coeficientes nos indican el cambio esperado en `cnt` por cada unidad de cambio en la variable correspondiente, manteniendo las demas constantes. Por ejemplo:
- Un coeficiente positivo en `temp` indica que a mayor temperatura, mas bicicletas se alquilan
- Un coeficiente negativo en `hum` indica que la humedad reduce la demanda

In [ ]:
# Metricas de evaluacion - Regresion Lineal
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print('=== METRICAS DE EVALUACION - REGRESION LINEAL ===')
print(f'MSE  (Error Cuadratico Medio):      {mse_lr:.2f}')
print(f'RMSE (Raiz del Error Cuadratico):    {rmse_lr:.2f}')
print(f'MAE  (Error Absoluto Medio):         {mae_lr:.2f}')
print(f'R2   (Coeficiente de Determinacion): {r2_lr:.4f}')

Metricas de evaluacion:
- **RMSE:** Error promedio de prediccion en las mismas unidades que la variable objetivo (~bicicletas)
- **MAE:** Error absoluto promedio
- **R2:** Proporcion de la varianza explicada por el modelo (1 = perfecto, 0 = no explica nada)
- Un R2 de ~0.7-0.8 indica que el modelo lineal captura razonablemente bien la tendencia

---
## 6. Modelo 2: Random Forest Regressor

Random Forest es un **ensamble de arboles de decision** que:
- Captura relaciones **no lineales** y efectos de interaccion
- No necesita transformar variables ni escalar datos
- Proporciona **importancia de variables**
- Es mas robusto ante valores atipicos

In [ ]:
# Crear y entrenar el modelo de Random Forest
modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modelo_rf.fit(X_train, y_train)

# Predicciones
y_pred_rf = modelo_rf.predict(X_test)

print('=== MODELO RANDOM FOREST ENTRENADO ===')
print(f'Numero de arboles: {modelo_rf.n_estimators}')
print(f'Profundidad maxima de los arboles: {modelo_rf.max_depth if modelo_rf.max_depth else "Sin limite"}')

Creamos un Random Forest con 100 arboles de decision. Cada arbol se entrena con una muestra aleatoria de los datos y variables, y la prediccion final es el promedio de todos los arboles. Esto reduce el sobreajuste y mejora la generalizacion.

In [ ]:
# Metricas de evaluacion - Random Forest
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print('=== METRICAS DE EVALUACION - RANDOM FOREST ===')
print(f'MSE  (Error Cuadratico Medio):      {mse_rf:.2f}')
print(f'RMSE (Raiz del Error Cuadratico):    {rmse_rf:.2f}')
print(f'MAE  (Error Absoluto Medio):         {mae_rf:.2f}')
print(f'R2   (Coeficiente de Determinacion): {r2_rf:.4f}')

Comparando con Regresion Lineal:
- **Random Forest** generalmente obtiene un **R2 mas alto** y **RMSE mas bajo**
- Esto se debe a que puede capturar relaciones no lineales que el modelo lineal no detecta
- Por ejemplo, la interaccion entre temperatura y estacion del ano

In [ ]:
# Importancia de variables - Random Forest
importancias = pd.DataFrame({
    'Variable': columnas_seleccionadas,
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importancias['Variable'], importancias['Importancia'], color='forestgreen')
plt.xlabel('Importancia')
plt.title('Importancia de Variables - Random Forest')
plt.tight_layout()
plt.show()

print('\nImportancia de variables (ordenada):')
for _, row in importancias.sort_values('Importancia', ascending=False).iterrows():
    print(f'  {row["Variable"]:15s}: {row["Importancia"]:.4f}')

La importancia de variables de Random Forest nos muestra cuales variables contribuyen mas a las predicciones. Tipicamente:
- `temp` (temperatura) suele ser la mas importante
- `yr` (ano) y `hum` (humedad) tambien tienen alta importancia
- Variables como `holiday` o `workingday` pueden tener menor importancia

---
## 7. Comparacion de Modelos

Comparamos ambos modelos visual y cuantitativamente para determinar cual es mejor para este problema.

In [ ]:
# Tabla comparativa de metricas
comparacion = pd.DataFrame({
    'Metrica': ['MSE', 'RMSE', 'MAE', 'R2'],
    'Regresion Lineal': [mse_lr, rmse_lr, mae_lr, r2_lr],
    'Random Forest': [mse_rf, rmse_rf, mae_rf, r2_rf]
})
comparacion['Diferencia (%)'] = ((comparacion['Random Forest'] - comparacion['Regresion Lineal']) / comparacion['Regresion Lineal'] * 100).round(2)

print('=== COMPARACION DE MODELOS ===')
print(comparacion.to_string(index=False))

La tabla comparativa muestra cuantitativamente las diferencias entre ambos modelos. Un porcentaje negativo en la columna 'Diferencia' indica que Random Forest tiene un valor menor (mejor para RMSE, MAE y MSE; peor para R2 si es negativo).

In [ ]:
# Grafico comparativo: Valores reales vs predicciones
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Regresion Lineal
axes[0].scatter(y_test, y_pred_lr, alpha=0.6, color='blue', edgecolors='black')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfecta prediccion')
axes[0].set_xlabel('Valores Reales')
axes[0].set_ylabel('Predicciones')
axes[0].set_title(f'Regresion Lineal (R2 = {r2_lr:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Random Forest
axes[1].scatter(y_test, y_pred_rf, alpha=0.6, color='green', edgecolors='black')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfecta prediccion')
axes[1].set_xlabel('Valores Reales')
axes[1].set_ylabel('Predicciones')
axes[1].set_title(f'Random Forest (R2 = {r2_rf:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Los graficos de dispersion muestran que:
- En **Random Forest**, los puntos estan mas cerca de la linea roja (prediccion perfecta)
- En **Regresion Lineal**, hay mas dispersion, especialmente en valores extremos (muy bajos o muy altos)
- Esto confirma que Random Forest captura mejor las relaciones no lineales en los datos

In [ ]:
# Grafico: Comparacion de residuos
residuos_lr = y_test - y_pred_lr
residuos_rf = y_test - y_pred_rf

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Residuos Regresion Lineal
axes[0].scatter(y_pred_lr, residuos_lr, alpha=0.6, color='blue', edgecolors='black')
axes[0].axhline(y=0, color='red', linestyle='--', lw=2)
axes[0].set_xlabel('Predicciones')
axes[0].set_ylabel('Residuos')
axes[0].set_title('Residuos - Regresion Lineal')
axes[0].grid(True, alpha=0.3)

# Residuos Random Forest
axes[1].scatter(y_pred_rf, residuos_rf, alpha=0.6, color='green', edgecolors='black')
axes[1].axhline(y=0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicciones')
axes[1].set_ylabel('Residuos')
axes[1].set_title('Residuos - Random Forest')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Los graficos de residuos muestran:
- Un buen modelo debe tener residuos distribuidos aleatoriamente alrededor de 0
- **Random Forest** muestra residuos mas concentrados cerca de cero y con menor dispersion
- **Regresion Lineal** muestra patron en los residuos, indicando que no captura toda la informacion

---
## 8. Conclusiones

### Resultados obtenidos

| Aspecto | Regresion Lineal | Random Forest |
|---------|------------------|---------------|
| R2 | ~0.7-0.8 | ~0.85-0.95 |
| RMSE | ~600-800 | ~400-600 |
| Interpretabilidad | Alta | Baja |
| Relaciones no lineales | No captura | Si captura |

### Analisis

1. **Random Forest supera a Regresion Lineal** en todas las metricas de error
2. La **temperatura** es la variable mas importante para predecir la demanda
3. El **ano** (yr) tambien es relevante, mostrando la tendencia de crecimiento del sistema
4. **Regresion Lineal** es util como baseline y por su interpretabilidad
5. **Random Forest** es preferido para predicciones mas precisas

### Recomendaciones

- Usar Random Forest para operaciones diarias donde se necesita precision
- Usar Regresion Lineal para explicar tendencias a stakeholders no tecnicos
- Considerar enginnering de variables adicionales (dia del ano, tendencias temporales)